PyTorch → TVM C Code Pipeline for AdaptViT
=================================================
Quantizes pretrained ViT models, compiles them to bare-metal C via TVM,
and generates the binary files needed for RISC-V deployment.

This notebook should be run after pruning and
linear probing, as it reads the pruned model
checkpoints and trained classifier heads saved by those scripts.

Pipeline:
  1. Load each pruned ViT, apply dynamic INT8 quantization, trace with TorchScript
  2. Convert to TVM Relay IR and compile to a C archive (.tar) per model
  3. Generate input image binary files (.bin) for two test samples
  4. Generate pruning mask binary files (.bin) for each sparsity level
  5. Generate classifier weight, bias, and scale binary files (.bin)
  6. Zip all deployment artifacts and download

Author: Vishnu PS

## Environment Setup

In [ ]:
!git clone https://github.com/WalterSimoncini/SnapViT.git 2>/dev/null || echo "Already cloned"
%cd SnapViT
!pip install -r /content/SnapViT/requirements.txt

In [ ]:
!pip install "https://files.pythonhosted.org/packages/9c/b2/adb56267ba6b6f05204407a7745e0c188e39589b7e33f490d4aa406d84e8/apache_tvm-0.14.dev273-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl"
!pip install numpy==1.26.4
!pip install torch torchvision
!pip install timm

In [ ]:
import os
from google.colab import drive

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
os.environ['TIMM_FUSED_ATTN'] = '0'
import torch
from pathlib import Path

BASE_PATH = '/content/drive/MyDrive/PhD_Works/SnapViT_Q2'
MODELS_FOLDER = f'{BASE_PATH}/model_outputs'

# Pruning levels to process for each model
PRUNING_CONFIGS = [
    (0.15, 0.0),
    (0.25, 0.1),
    (0.35, 0.2),
    (0.45, 0.3),
    (0.55, 0.4),
    (0.65, 0.5),
]

TIMM_MODEL_NAMES = {
    'vit_tiny_r':    'timm/vit_tiny_r_s16_p8_224.augreg_in21k_ft_in1k',
    'augreg_vits16': 'timm/vit_small_patch16_224.augreg_in21k_ft_in1k',
    'augreg_vitb16': 'timm/vit_base_patch16_224.augreg_in21k_ft_in1k',
    'dino_vits16':   'timm/vit_small_patch16_224.dino',
    'dino_vitb16':   'timm/vit_base_patch16_224.dino',
    'deit3_vits16':  'timm/deit3_small_patch16_224.fb_in22k_ft_in1k',
    'deit3_vitb16':  'timm/deit3_base_patch16_224.fb_in22k_ft_in1k',
}

DATASET_PATH = "/content/local_pets"

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model_folders = [d for d in os.listdir(MODELS_FOLDER)
                 if os.path.isdir(os.path.join(MODELS_FOLDER, d)) and d in TIMM_MODEL_NAMES]

print(f"Found {len(model_folders)} models to process: {model_folders}")
print(f"Device: {device}\n")

In [ ]:
import os
import re

LOCAL_IMAGES = "/content/local_pets"
RAW_DIR      = "/tmp/pets_raw"
TAR_PATH     = "/tmp/pets.tar.gz"

os.makedirs(LOCAL_IMAGES, exist_ok=True)

# Check if already organised from a previous cell run this session
existing_breeds = [d for d in os.listdir(LOCAL_IMAGES)
                   if os.path.isdir(os.path.join(LOCAL_IMAGES, d))]

if len(existing_breeds) >= 37:
    print(f"Already organised this session ({len(existing_breeds)} breeds) — skipping")
else:
    print("Downloading Oxford-IIIT Pets images (~800 MB)...")
    !wget -q --show-progress -O {TAR_PATH} \
        https://thor.robots.ox.ac.uk/~vgg/data/pets/images.tar.gz

    print("Extracting...")
    os.makedirs(RAW_DIR, exist_ok=True)
    !tar -xf {TAR_PATH} -C {RAW_DIR} --strip-components=1

    print("Organising into breed subfolders...")
    pattern = re.compile(r'^(.+)_\d+\.jpg$', re.IGNORECASE)
    for fname in os.listdir(RAW_DIR):
        m = pattern.match(fname)
        if not m:
            continue
        breed = m.group(1)
        breed_dir = os.path.join(LOCAL_IMAGES, breed)
        os.makedirs(breed_dir, exist_ok=True)
        src = os.path.join(RAW_DIR, fname)
        dst = os.path.join(breed_dir, fname)
        if not os.path.exists(dst):
            os.rename(src, dst)

    breeds       = [d for d in os.listdir(LOCAL_IMAGES)
                    if os.path.isdir(os.path.join(LOCAL_IMAGES, d))]
    total_images = sum(len(os.listdir(os.path.join(LOCAL_IMAGES, b))) for b in breeds)
    print(f"\nDATASET READY!")
    print(f"Breeds   : {len(breeds)}")
    print(f"Images   : {total_images}")
    print(f"Location : {LOCAL_IMAGES}")

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset, random_split
import random
import sys
import os
from tqdm import tqdm

# ── Device Setup ─────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}\n")

# ← USING LOCAL SSD PATH
IMAGES_PATH = LOCAL_IMAGES

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# Load full dataset
dataset = datasets.ImageFolder(IMAGES_PATH, transform=transform)

# === FIXED REPRODUCIBLE TEST SPLIT (20%) ===
indices = list(range(len(dataset)))
random.seed(2026)                    # Fixed seed → always same split
random.shuffle(indices)

test_size = int(0.2 * len(dataset))
trainval_indices = indices[:-test_size]
test_indices     = indices[-test_size:]

trainval_dataset = Subset(dataset, trainval_indices)
test_dataset     = Subset(dataset, test_indices)

# Split trainval into train (80%) + val (20%)
train_size = int(0.8 * len(trainval_dataset))
val_size   = len(trainval_dataset) - train_size

train_set, val_set = random_split(
    trainval_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

# DataLoaders
train_loader = DataLoader(train_set, batch_size=128, shuffle=True,
                          num_workers=8, pin_memory=True, persistent_workers=True, prefetch_factor=4)
val_loader   = DataLoader(val_set,   batch_size=128, shuffle=False,
                          num_workers=8, pin_memory=True, persistent_workers=True)
test_loader  = DataLoader(test_dataset, batch_size=128, shuffle=False,
                          num_workers=8, pin_memory=True, persistent_workers=True)

print(f"Total images     : {len(dataset)}")
print(f"Train            : {len(train_set)}")
print(f"Validation       : {len(val_set)}")
print(f"Test (held-out)  : {len(test_dataset)}")
print(f"Classes          : {len(dataset.classes)}")

CLASS_NAMES = dataset.classes
NUM_CLASSES = len(CLASS_NAMES)

## TVM compilation — one .tar per model
Each model goes through: dynamic INT8 quantization → TorchScript trace →
TVM Relay frontend → AOT compilation targeting bare-metal C (CRT runtime).

CPU is used throughout (device = cpu) — TVM's C backend requires the model
to be traced on CPU. The resulting .tar contains the generated C source,
weights baked into a const workspace

TIMM_FUSED_ATTN=0 is set at the top to disable fused attention kernels,
which are not traceable by TorchScript.

In [ ]:
import os
import torch
import torch.nn as nn
import torch.quantization as quantization
import timm
import tvm
from tvm import relay
# import tvm.relay as relay
from pathlib import Path

BASE_PATH = '/content/drive/MyDrive/PhD_Works/SnapViT_Q2'
MODELS_FOLDER = f'{BASE_PATH}/model_outputs'

MLP_PRUNING_LEVEL = 0.15
HEAD_PRUNING_LEVEL = 0.0

device = torch.device('cpu')

TIMM_MODEL_NAMES = {
    'vit_tiny_r':    'timm/vit_tiny_r_s16_p8_224.augreg_in21k_ft_in1k',
    'augreg_vits16': 'timm/vit_small_patch16_224.augreg_in21k_ft_in1k',
    'augreg_vitb16': 'timm/vit_base_patch16_224.augreg_in21k_ft_in1k',
    'dino_vits16':   'timm/vit_small_patch16_224.dino',
    'dino_vitb16':   'timm/vit_base_patch16_224.dino',
    'deit3_vits16':  'timm/deit3_small_patch16_224.fb_in22k_ft_in1k',
    'deit3_vitb16':  'timm/deit3_base_patch16_224.fb_in22k_ft_in1k',
}

model_folders = sorted([d for d in os.listdir(MODELS_FOLDER)
                        if os.path.isdir(os.path.join(MODELS_FOLDER, d)) and d in TIMM_MODEL_NAMES])

print(f"Found {len(model_folders)} models: {model_folders}")
print(f"Using FIXED pruning ratio → MLP={MLP_PRUNING_LEVEL}, Head={HEAD_PRUNING_LEVEL}")
print(f"Device: CPU (required for TVM compatibility)\n")

DATASET_PATH = "/content/local_pets"
CLASS_NAMES = sorted([
    d for d in os.listdir(DATASET_PATH)
    if os.path.isdir(os.path.join(DATASET_PATH, d))
])
NUM_CLASSES = len(CLASS_NAMES)
print(f"Number of classes: {NUM_CLASSES}\n")


def process_quantized_model(model_name: str):
    print(f"\n{'='*90}")
    print(f"Processing: {model_name}")
    print(f"{'='*90}\n")

    head_path = os.path.join(MODELS_FOLDER, model_name, "head_mlp_base.pt")

    if not os.path.exists(head_path):
        print(f"  head_mlp_base.pt not found: {head_path}")
        return False

    OUTPUT_NAME = f"{model_name}"
    TAR_PATH = f"/content/{OUTPUT_NAME}.tar"

    try:
        print(f"  Loading base model and applying quantization...\n")

        timm_name = TIMM_MODEL_NAMES[model_name]
        model_for_export = timm.create_model(timm_name, pretrained=True, num_classes=0)
        model_for_export.head = nn.Linear(model_for_export.num_features, NUM_CLASSES)
        head_state = torch.load(head_path, map_location='cpu')
        model_for_export.head.load_state_dict(head_state, strict=False)
        model_for_export = model_for_export.eval().cpu()

        # Apply dynamic quantization
        model_for_export = quantization.quantize_dynamic(
            model_for_export,
            qconfig_spec={torch.nn.Linear, torch.nn.Conv2d},
            dtype=torch.qint8,
        )

        print(f"  Base model loaded and quantized on CPU\n")

        # Verify quantization
        quant_linear_count = sum(1 for m in model_for_export.modules()
                                if isinstance(m, torch.nn.quantized.Linear))
        quant_conv_count = sum(1 for m in model_for_export.modules()
                              if isinstance(m, torch.nn.quantized.Conv2d))

        print(f"  Quantized Linear layers: {quant_linear_count}")
        print(f"  Quantized Conv2d layers: {quant_conv_count}\n")

        # Trace the quantized model
        print(f"  Tracing quantized model with torch.jit...\n")

        dummy_input = torch.randn(1, 3, 224, 224)

        with torch.no_grad():
            traced_model = torch.jit.trace(model_for_export, dummy_input, check_trace=False)

        print(f" Tracing successful\n")

        # Trace verification
        with torch.no_grad():
            out_orig = model_for_export(dummy_input)
            out_traced = traced_model(dummy_input)
        trace_diff = torch.abs(out_orig - out_traced).max().item()
        print(f"  Trace verification: max diff = {trace_diff:.8f}\n")

        # Convert to TVM Relay
        print(f"  Converting to TVM Relay...\n")

        mod, params = relay.frontend.from_pytorch(
            traced_model,
            [("input0", (1, 3, 224, 224))]
        )

        mod["main"] = relay.build_module.bind_params_by_name(mod["main"], params)

        print(f"  Conversion successful | Parameters: {len(params)}\n")

        # Compile to C
        print(f"  Compiling to C code...\n")

        target = tvm.target.Target("c")
        runtime = relay.backend.Runtime("crt")
        executor = relay.backend.Executor("aot", {
            "unpacked-api": True,
            "interface-api": "c",
            "workspace-byte-alignment": 8,
        })

        with tvm.transform.PassContext(opt_level=3, config={"tir.disable_vectorize": True}):
            lib = relay.build(
                mod,
                target=target,
                runtime=runtime,
                executor=executor
            )

        print(f"  Compilation successful\n")

        # Step 6: Export to tar
        print(f"  Exporting to tar archive...\n")

        os.makedirs(os.path.dirname(TAR_PATH), exist_ok=True)
        tvm.micro.export_model_library_format(lib, TAR_PATH)

        tar_size_mb = os.path.getsize(TAR_PATH) / (1024 * 1024)
        print(f"  Export successful")
        print(f"    File: {TAR_PATH}")
        print(f"    Size: {tar_size_mb:.2f} MB\n")

        # Save TAR to model folder
        print(f"  Saving to model folder...\n")
        import shutil
        final_tar_name = f"{OUTPUT_NAME}.tar"
        model_tar_path = os.path.join(MODELS_FOLDER, model_name, final_tar_name)

        shutil.copy2(TAR_PATH, model_tar_path)

        print(f"  TAR file saved")
        print(f"  Location: {model_tar_path}\n")

        print(f" Model {model_name} C code generation completed.\n")
        return True

    except Exception as e:
        print(f"  Error processing {model_name}: {e}\n")
        import traceback
        traceback.print_exc()
        return False


# Main Loop
print("Starting processing for all quantized models...\n")

results = {}
for model_name in model_folders:
    results[model_name] = process_quantized_model(model_name)

print("\n" + "="*95)
print("SECTION 3+ COMPLETE")
print("="*95)
print(f"\nResults:")
for model_name, success in results.items():
    status = "✓ SUCCESS" if success else "✗ FAILED"
    print(f"  {model_name:<20s} {status}")

successful = sum(1 for v in results.values() if v)
failed = sum(1 for v in results.values() if not v)
print(f"\nSummary: {successful} successful, {failed} failed")
print(f"Fixed pruning ratio: MLP={MLP_PRUNING_LEVEL}, Head={HEAD_PRUNING_LEVEL}")
print("Only .tar files were saved directly inside each model's folder.")
print("="*95)


## Input image binary files
Two sample images are preprocessed (resize → normalize → flatten) and saved
as raw float32 binary files. These are used as fixed test inputs when running
C inference on the RISC-V simulator, so results can be cross-checked against
PyTorch output.

In [ ]:
import os
from PIL import Image, ImageFile
import numpy as np
from torchvision import transforms # Re-import transforms
import torch

ImageFile.LOAD_TRUNCATED_IMAGES = True

# Redefine transform for image preprocessing
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# Ensure device is correctly set, assuming CPU for TVM compatibility as per previous cells
device = torch.device('cpu')

# Sample images to generate binaries for
TEST_CASES = {
    "russian_blue":      f"{DATASET_PATH}/Russian_Blue/Russian_Blue_10.jpg",
    "british_shorthair": f"{DATASET_PATH}/British_Shorthair/British_Shorthair_89.jpg",
}

# Common output directory — sits outside all model/pruning folders
GLOBAL_HEADERS_DIR = os.path.join(BASE_PATH, "model_outputs")
os.makedirs(GLOBAL_HEADERS_DIR, exist_ok=True)

print(f"Generating input image binary files...")
print(f"Output directory: {GLOBAL_HEADERS_DIR}\n")

for label, img_path in TEST_CASES.items():
    if not os.path.exists(img_path):
        print(f"  Skipping {label} — file not found: {img_path}")
        continue

    # Load and preprocess image
    img = Image.open(img_path).convert("RGB")
    input_tensor = transform(img).unsqueeze(0).to(device)
    input_np = input_tensor.cpu().numpy().astype(np.float32)  # [1, 3, 224, 224]

    # Write raw binary
    bin_path = os.path.join(GLOBAL_HEADERS_DIR, f"input_image_{label}.bin")
    input_np.flatten().tofile(bin_path)

    size_kb = os.path.getsize(bin_path) / 1024
    print(f"  ✓ {os.path.basename(bin_path)}  |  shape: {input_np.shape}  |  {size_kb:.1f} KB")

print("\n" + "="*80)
print("INPUT IMAGE BINARY GENERATION COMPLETED")
print("="*80)
print(f"Output directory: {GLOBAL_HEADERS_DIR}")
print(f"Expected size per image: 1 x 3 x 224 x 224 x 4 bytes = "
      f"{1*3*224*224*4/1024:.1f} KB")

In [ ]:
def get_pruning_suffix(mlp_ratio: float, head_ratio: float) -> str:
    """Convert (0.15, 0.0) → 'mlp015_head000' (C-friendly identifier)"""
    mlp_str = f"{int(mlp_ratio * 100):03d}"
    head_str = f"{int(head_ratio * 100):03d}"
    return f"mlp{mlp_str}_head{head_str}"

## Pruning mask binary files
Converts pruning_masks.pt (saved by the pruning notebook) into raw float32
binary files loadable by the RISC-V runtime.
Two sets of files are generated:
  - Pruned masks: one mlp_masks.bin + head_masks.bin per sparsity level
  - Full masks  : all-ones mask versions written once per model (used for the
                  checking code edit overhead)

In [ ]:
import os
import numpy as np
import torch

print("=== Pruning Mask Binary File Generation (All Levels) ===\n")


def write_binary_file(path, numpy_array, dtype=None):
    if dtype is not None:
        numpy_array = numpy_array.astype(dtype)

    numpy_array.flatten().tofile(path)
    size_kb = os.path.getsize(path) / 1024
    print(f"      → {os.path.basename(path)} ({size_kb:.1f} KB, dtype={numpy_array.dtype})")

PRUNING_LEVELS = [
    (0.15, 0.0),
    (0.25, 0.1),
    (0.35, 0.2),
    (0.45, 0.3),
    (0.55, 0.4),
    (0.65, 0.5),
]

print(f"Models to process: {model_folders}")
print(f"Pruning levels   : {len(PRUNING_LEVELS)}\n")

total_generated = 0

for model_name in model_folders:
    print(f"\n{'#'*70}")
    print(f"Model: {model_name}")
    print(f"{'#'*70}")

    full_masks_written = False  # Write full masks once per model

    for mlp_level, head_level in PRUNING_LEVELS:
        pruning_folder = f"mlp-{mlp_level}-heads-{head_level}"
        folder_path    = os.path.join(MODELS_FOLDER, model_name, pruning_folder)
        masks_path     = os.path.join(folder_path, 'pruning_masks.pt')

        if not os.path.exists(masks_path):
            print(f"  [{pruning_folder}] pruning_masks.pt not found — skipping")
            continue

        # Load pruning masks
        masks = torch.load(masks_path, map_location='cpu', weights_only=False)
        mlp_masks_np  = masks['mlp_masks'].numpy().astype(np.float32)
        head_masks_np = masks['head_masks'].numpy().astype(np.float32)

        print(f"  [{pruning_folder}]")
        print(f"    MLP mask shape : {mlp_masks_np.shape}")
        print(f"    Head mask shape: {head_masks_np.shape}")

        write_binary_file(
            os.path.join(folder_path, 'mlp_masks.bin'),
            mlp_masks_np
        )
        write_binary_file(
            os.path.join(folder_path, 'head_masks.bin'),
            head_masks_np
        )

        total_generated += 2

        if not full_masks_written:
            mlp_masks_full_np  = np.ones_like(mlp_masks_np,  dtype=np.float32)
            head_masks_full_np = np.ones_like(head_masks_np, dtype=np.float32)

            model_root = os.path.join(MODELS_FOLDER, model_name)

            print(f"\n  [full masks — level 0]")
            write_binary_file(
                os.path.join(model_root, 'mlp_masks_full.bin'),
                mlp_masks_full_np
            )
            write_binary_file(
                os.path.join(model_root, 'head_masks_full.bin'),
                head_masks_full_np
            )

            total_generated  += 2
            full_masks_written = True

        print(f"    ✓ Done")

print("\n" + "="*80)
print("MASK BINARY GENERATION COMPLETED")
print("="*80)
print(f"Total binary files generated: {total_generated}")
print("\n  Per pruning level folder:")
print("    mlp_masks.bin        — pruned mask")
print("    head_masks.bin       — pruned mask")
print("\n  Per model root folder (once only):")
print("    mlp_masks_full.bin   — all-ones mask for level 0")
print("    head_masks_full.bin  — all-ones mask for level 0")

## Classifier weight, bias, and scale binary files
The base classification head is baked into the TVM .tar. For each pruning level, the corresponding classifier is injected
at runtime via memcpy rewriting the workspace. This allows the same
compiled binary to serve all sparsity levels by simply swapping the head.

Weights are quantized per-tensor to INT16 but with INT8 scale (scale = max(|w|) / 127), this reproduce TVMs widening behaviour. Finally saved as classifier_weights.bin. Biases remain float32. The per-level weight scales are also collected into a single classifier_scales.bin file so the C runtime can dequantize correctly for each active configuration.

In [ ]:
import os
import torch
import numpy as np

PRUNING_CONFIGS = [
    (0.15, 0.0),
    (0.25, 0.1),
    (0.35, 0.2),
    (0.45, 0.3),
    (0.55, 0.4),
    (0.65, 0.5),
]

print(f"Models to process : {model_folders}")
print(f"Pruning levels    : {len(PRUNING_CONFIGS)}\n")

total_generated = 0

for model_name in model_folders:
    print(f"\n{'#'*70}")
    print(f"Model: {model_name}")
    print(f"{'#'*70}")

    if model_name == 'vit_tiny_r':
        EMBED_DIM = 192
    elif model_name in ['augreg_vits16', 'dino_vits16', 'deit3_vits16']:
        EMBED_DIM = 384
    elif model_name in ['augreg_vitb16', 'dino_vitb16', 'deit3_vitb16']:
        EMBED_DIM = 768
    else:
        EMBED_DIM = 384

    all_scales = []

    for mlp_ratio, head_ratio in PRUNING_CONFIGS:
        pruning_folder    = f"mlp-{mlp_ratio}-heads-{head_ratio}"
        classifier_folder = os.path.join(MODELS_FOLDER, model_name, pruning_folder)
        head_path         = os.path.join(classifier_folder, "head_mlp.pt")

        if not os.path.exists(head_path):
            print(f"  [{pruning_folder}] head_mlp.pt not found — skipping")
            continue

        print(f"  [{pruning_folder}]")

        try:
            state_dict = torch.load(head_path, map_location='cpu')

            if 'weight' not in state_dict or 'bias' not in state_dict:
                print(f"    ✗ Missing weight/bias keys — skipping")
                continue

            w_fp32 = state_dict['weight'].detach().float().cpu().numpy().astype(np.float32)
            b_fp32 = state_dict['bias'].detach().float().cpu().numpy().astype(np.float32)

            print(f"    Weights shape : {tuple(w_fp32.shape)}  Bias shape: {tuple(b_fp32.shape)}")
            print(f"    FP32 range    : [{w_fp32.min():.6f}, {w_fp32.max():.6f}]")

            # Per-tensor symmetric int16 quantization
            scale = np.abs(w_fp32).max() / 127.0
            w_int16 = (w_fp32 / scale).round().astype(np.int16)
            w_int16 = np.clip(w_int16, -128, 127).astype(np.int16)
            print(f"    Scale         : {scale:.10f}")
            print(f"    INT16 range   : [{w_int16.min()}, {w_int16.max()}]")

            # Store scale for later
            all_scales.append(scale)

            flat = w_int16.flatten()
            print(f"    First 8 (hex) : " +
                  "  ".join(f"{'+' if v >= 0 else ''}{v:#06x}" for v in flat[:8]))

            assert w_int16.dtype == np.int16, f"ERROR: Expected int16, got {w_int16.dtype}"

            write_binary_file(
                os.path.join(classifier_folder, 'classifier_weights.bin'),
                w_int16.reshape(NUM_CLASSES, EMBED_DIM)
            )

            assert b_fp32.dtype == np.float32, f"ERROR: Expected float32, got {b_fp32.dtype}"

            write_binary_file(
                os.path.join(classifier_folder, 'classifier_bias.bin'),
                b_fp32
            )

            total_generated += 2
            print(f"    ✓ classifier_weights.bin  (int16, [{NUM_CLASSES} × {EMBED_DIM}])")
            print(f"    ✓ classifier_bias.bin     (float32, [{NUM_CLASSES}])")

        except Exception as e:
            print(f"    ✗ Error: {str(e)}")
            import traceback
            traceback.print_exc()

    if len(all_scales) == 6:
        scales_array = np.array(all_scales, dtype=np.float32)
        scales_path = os.path.join(MODELS_FOLDER, model_name, 'classifier_scales.bin')

        scales_array.tofile(scales_path)

        print(f"\n  ✓ classifier_scales.bin generated")
        print(f"    Path: {scales_path}")
        print(f"    Size: {os.path.getsize(scales_path)} bytes (6 float32 values)")
        print(f"    Scales:")
        for i, (config, scale) in enumerate(zip(PRUNING_CONFIGS, all_scales)):
            print(f"      Level {i+1} (mlp-{config[0]}-heads-{config[1]}): {scale:.10f}")

        total_generated += 1
    else:
        print(f"\n  ✗ Could not generate classifier_scales.bin (only {len(all_scales)}/6 scales found)")

print("\n" + "="*85)
print("CLASSIFIER BINARY GENERATION COMPLETED")
print("="*85)
print(f"Total binary files generated : {total_generated}")
print("  Per model:")
print("    classifier_scales.bin      — float32 [6 scales for 6 pruning levels]")
print("  Per pruning level folder:")
print("    classifier_weights.bin  — int16 [NUM_CLASSES × EMBED_DIM]")
print("    classifier_bias.bin     — float32 [NUM_CLASSES]")
print("\nQuantisation: scale = max(|w|) / 32767  (per-tensor symmetric int16)")
print("C code loads scales from classifier_scales.bin for dequantization.")
print("="*85)

## Package deployment artifacts
Collects all .bin and .tar files into a single zip preserving the
model_outputs/ folder structure, then downloads it from Colab.
This zip is the complete input to the ASIP Designer compilation step.

In [ ]:
import os
import zipfile
from google.colab import files

BASE_PATH  = '/content/drive/MyDrive/PhD_Works/SnapViT_Q2'
SOURCE_DIR = os.path.join(BASE_PATH, 'model_outputs')

ZIP_NAME = "/content/model_outputs.zip"

print("Creating zip with .bin and .tar files...\n")

included_count = 0
skipped_count  = 0

with zipfile.ZipFile(ZIP_NAME, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, file_list in os.walk(SOURCE_DIR):
        for file in file_list:

            if file.endswith('.tar') or file.endswith('.bin'):
                include = True

            else:
                include = False
                skipped_count += 1

            if include:
                file_path = os.path.join(root, file)
                arcname   = os.path.relpath(file_path, BASE_PATH)
                zipf.write(file_path, arcname)
                included_count += 1

                if included_count % 20 == 0:
                    print(f"  Added {included_count} files so far...")

print("\n" + "="*80)
print("ZIP CREATION COMPLETED")
print("="*80)
print(f"Files included : {included_count}  (.bin + .tar)")
print(f"Files skipped  : {skipped_count}  (old .h headers and other files)")
print(f"Zip created    : {ZIP_NAME}")
print(f"Zip size       : {os.path.getsize(ZIP_NAME) / (1024*1024):.2f} MB")
print("\nFolder structure preserved from 'model_outputs/'")
print("\nExpected contents per pruning level folder:")
print("  mlp_masks.bin")
print("  head_masks.bin")
print("  classifier_weights.bin")
print("  classifier_bias.bin")
print("\nExpected in model root folder:")
print("  <model_name>.tar")
print("\nExpected in model_outputs root:")
print("  input_image_russian_blue.bin")
print("  input_image_british_shorthair.bin")

print("\nDownloading zip file...")
files.download(ZIP_NAME)

print("\n✓ Done!")